# Clase práctica 2: Tipos de variables, tablas de frecuencia y gráficos
## Estadística Descriptiva · Introducción a R (continuación)

En la clase práctica 1 identificamos qué es una **población**, qué es una **muestra**, y aprendimos a clasificar variables. Hoy damos el siguiente paso:

1. Repasamos y **ampliamos la clasificación de variables**.
2. Calculamos **frecuencia absoluta, relativa, acumulada y relativa acumulada** — tanto para una variable cualitativa como para una **cuantitativa continua agrupada en intervalos**.
3. Creamos una **variable categórica nueva** a partir de la diferencia entre dos años (2025 y 2026) del Índice Departamental de Competitividad (IDC), y vemos cómo se distribuyen los departamentos en esas categorías.
4. Construimos **gráficos de barras, de torta (pastel) e histograma** — cada uno apropiado para un tipo de variable distinto.

Cada celda de código va acompañada de una explicación de **qué hace cada función de R que usamos**, para que puedas reutilizarlas en tus propios ejercicios.

## 0. Preparar el entorno

Recordatorio rápido (ver Clase práctica 1 para el detalle):

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → R**.
2. Sube `idc_2025_2026.csv` con el panel de archivos 📁 de la izquierda.

Cargamos los datos igual que en la clase anterior:

- `read.csv()` lee un archivo de texto separado por comas y lo convierte en un `data.frame` (una tabla) de R.
- El argumento `encoding = "UTF-8"` le indica a R cómo interpretar los caracteres especiales (tildes, "ñ") del archivo.

In [ ]:
idc <- read.csv("idc_2025_2026.csv", encoding = "UTF-8")


`str()` (*structure*) y `dim()` nos recuerdan, de un vistazo, cuántas filas/columnas tiene el conjunto de datos y qué tipo de dato tiene cada columna.

In [ ]:
str(idc)
dim(idc)


## 1. Repaso ampliado: tipos de variables

Recordemos la clasificación:

- **Cualitativa nominal:** categorías sin orden (`departamento`, `region`).
- **Cualitativa ordinal:** categorías con un orden natural (la variable que vamos a crear hoy, `variacion_categoria`, será un ejemplo).
- **Cuantitativa continua:** resulta de una medición, puede tomar cualquier valor en un rango (`idc_2025`, `idc_2026`, y todos los puntajes por pilar).
- **Cuantitativa discreta / posicional:** números enteros que representan un conteo o una posición (`rank_2025`, `rank_2026`, `cambio_pos`).

`sapply()` aplica una misma función a cada columna de `idc` y devuelve el resultado en un vector. Aquí la usamos junto con `class()` para ver cómo interpreta R cada columna:

In [ ]:
sapply(idc, class)


`table()` sobre ese resultado nos dice **cuántas** columnas hay de cada tipo (`character` = texto/cualitativa, `numeric` = cuantitativa continua, `integer` = cuantitativa discreta):

In [ ]:
table(sapply(idc, class))


## 2. Tablas de frecuencia para una variable cualitativa

Repasemos las cuatro frecuencias que vimos en la teoría, aplicadas a la variable `region` (cualitativa nominal):

- **Frecuencia absoluta ($f$):** número de departamentos en cada región.
- **Frecuencia relativa ($h$):** proporción del total que representa cada región ($h = f/n$).
- **Frecuencia acumulada ($F$):** suma progresiva de las frecuencias absolutas.
- **Frecuencia relativa acumulada ($H$):** suma progresiva de las frecuencias relativas.

`table()` cuenta cuántas veces aparece cada categoría de una variable — es la función clave para la frecuencia absoluta.

In [ ]:
f_region <- table(idc$region)
f_region


`prop.table()` convierte una tabla de frecuencias absolutas en frecuencias **relativas** (proporciones que suman 1). Con `round(..., 3)` la redondeamos a 3 decimales para que sea más legible.

In [ ]:
h_region <- round(prop.table(f_region), 3)
h_region


`cumsum()` (*cumulative sum*) calcula la suma acumulada de un vector, elemento por elemento. Aplicada a la tabla de frecuencias, nos da la frecuencia acumulada.

In [ ]:
F_region <- cumsum(f_region)
H_region <- cumsum(h_region)
F_region
H_region


> **Nota conceptual:** `region` es una variable **nominal** — sus categorías no tienen un orden natural (¿por qué "Caribe" debería ir antes o después de "Pacífico"?). Por eso, aunque R nos calcule la frecuencia acumulada sin problema, **su interpretación no es muy útil aquí**: la acumulada solo tiene sentido pleno cuando existe un orden lógico entre las categorías — como en los datos agrupados en intervalos que veremos a continuación.

Armemos una tabla resumen con las cuatro frecuencias juntas, usando `data.frame()` (que construye una tabla combinando varios vectores en columnas):

In [ ]:
tabla_region <- data.frame(
  region = names(f_region),
  f = as.numeric(f_region),
  h = as.numeric(h_region),
  F = as.numeric(F_region),
  H = as.numeric(H_region)
)
tabla_region


## 3. Tabla de frecuencias para datos continuos agrupados en intervalos

Cuando la variable es **cuantitativa continua**, como `idc_2026`, no tiene sentido contar cuántas veces se repite cada valor exacto (probablemente ninguno se repite). En su lugar, seguimos el procedimiento de datos agrupados que vimos en las diapositivas de la Unidad 1:

1. **Rango ($R$):** diferencia entre el valor máximo y el mínimo.
2. **Número de clases ($m$):** aplicando la **regla de Sturges**, $m = 1 + 3.3 \log_{10}(n)$, redondeado al entero más cercano.
3. **Amplitud de clase ($C$):** $C = R / m$.
4. **Marca de clase:** punto medio de cada intervalo — representa a toda la clase en cálculos posteriores.

Empecemos calculando $n$, el mínimo, el máximo y el rango con las funciones básicas `length()`, `min()` y `max()`:

In [ ]:
n <- length(idc$idc_2026)
minimo <- min(idc$idc_2026)
maximo <- max(idc$idc_2026)
R <- maximo - minimo

n
minimo
maximo
R


Ahora calculamos $m$ con la regla de Sturges. `log10()` es el logaritmo en base 10, y `round()` redondea al entero más cercano (necesitamos un número entero de clases).

In [ ]:
m <- round(1 + 3.3 * log10(n))
m


La amplitud de cada clase, redondeada a 2 decimales para trabajar con un número más manejable:

In [ ]:
C <- round(R / m, 2)
C


Para no dejar el valor mínimo justo en el borde de la primera clase, empezamos un poco antes: redondeamos el mínimo hacia abajo a un decimal con `floor()` (que siempre redondea hacia abajo). `seq()` genera una secuencia de números igualmente espaciados — aquí construye los $m+1$ límites de los $m$ intervalos, empezando en `inicio` y avanzando de a `C`.

In [ ]:
inicio <- floor(minimo * 10) / 10
cortes <- seq(inicio, by = C, length.out = m + 1)
cortes


`cut()` —ya la usamos en la clase pasada para crear categorías a mano; aquí la usamos con los `breaks` (límites) que acabamos de calcular, para agrupar automáticamente cada valor de `idc_2026` en su intervalo correspondiente. `include.lowest = TRUE` asegura que el valor mínimo quede incluido en la primera clase.

In [ ]:
clases_idc <- cut(idc$idc_2026, breaks = cortes, include.lowest = TRUE)

f_idc <- table(clases_idc)
f_idc


Calculamos la marca de clase de cada intervalo (el punto medio): tomamos todos los límites **menos el último** (`cortes[-length(cortes)]`) y le sumamos la mitad de la amplitud.

In [ ]:
marca_clase <- cortes[-length(cortes)] + C / 2
marca_clase


Y armamos la tabla completa de frecuencias para datos agrupados, igual que hicimos con `region`, pero agregando la columna de marca de clase:

In [ ]:
h_idc <- round(prop.table(f_idc), 3)
F_idc <- cumsum(f_idc)
H_idc <- cumsum(h_idc)

tabla_idc <- data.frame(
  intervalo = names(f_idc),
  marca_clase = round(marca_clase, 2),
  f = as.numeric(f_idc),
  h = as.numeric(h_idc),
  F = as.numeric(F_idc),
  H = as.numeric(H_idc)
)
tabla_idc


> **Nota:** la columna `H` puede terminar en 0.999 en vez de 1.000 — es normal, es solo el efecto acumulado de redondear `h` a 3 decimales en cada fila, no un error de cálculo.

Con esta tabla ya podríamos, por ejemplo, decir: "el 60.6% de los departamentos tiene un IDC 2026 de 5.38 o menos" (leyendo la fila correspondiente en la columna `H`).

## 4. Crear una variable categórica a partir de la variación 2025-2026

Vamos a construir una nueva variable: **cuánto cambió el IDC de cada departamento entre 2025 y 2026**, clasificado en tres categorías. Esto convierte una variable cuantitativa continua (la diferencia) en una variable **cualitativa ordinal** (con un orden lógico: *Retrocedió* < *Estable* < *Mejoró*).

Primero calculamos la diferencia. El operador `$` accede a una columna del `data.frame`; al asignar con `<-` a `idc$diferencia_idc`, creamos una columna **nueva**.

In [ ]:
idc$diferencia_idc <- idc$idc_2026 - idc$idc_2025


`summary()` da un resumen rápido de cinco números (mínimo, cuartiles, máximo) más la media — útil para decidir dónde poner los puntos de corte de nuestra clasificación.

In [ ]:
summary(idc$diferencia_idc)


Con ese resumen, definimos tres categorías:

- **Retrocedió:** el IDC bajó más de 0.05 puntos.
- **Estable:** el IDC se mantuvo entre -0.05 y +0.05 puntos (un cambio pequeño, dentro del "ruido" normal de una medición año a año).
- **Mejoró:** el IDC subió más de 0.05 puntos.

Esta vez usamos `cut()` con puntos de corte elegidos **a criterio del analista** (no con la regla de Sturges, que es para el número de clases de una tabla, no para categorías con significado propio como "mejoró"/"empeoró"). `-Inf` e `Inf` representan "menos infinito" e "infinito", para no dejar ningún valor por fuera; `labels` les da nombre a las categorías resultantes.

In [ ]:
idc$variacion_categoria <- cut(
  idc$diferencia_idc,
  breaks = c(-Inf, -0.05, 0.05, Inf),
  labels = c("Retrocedió", "Estable", "Mejoró")
)

table(idc$variacion_categoria)


## 5. Frecuencias completas de la nueva variable

A diferencia de `region`, `variacion_categoria` **sí tiene un orden lógico** (de peor a mejor desempeño). Aquí la frecuencia acumulada cobra sentido real: por ejemplo, `F` en "Estable" nos dice cuántos departamentos *no mejoraron* (retrocedieron o se mantuvieron estables).

In [ ]:
f_var <- table(idc$variacion_categoria)
h_var <- round(prop.table(f_var), 3)
F_var <- cumsum(f_var)
H_var <- cumsum(h_var)

tabla_variacion <- data.frame(
  categoria = names(f_var),
  f = as.numeric(f_var),
  h = as.numeric(h_var),
  F = as.numeric(F_var),
  H = as.numeric(H_var)
)
tabla_variacion


## 6. ¿Cómo se organizan los departamentos según esta nueva variable?

`order()` devuelve las posiciones que reordenan un vector; usado dentro de `idc[ ... , ]` reordena las **filas** completas del `data.frame`. Aquí ordenamos primero por la categoría y luego alfabéticamente por departamento dentro de cada categoría.

In [ ]:
idc[order(idc$variacion_categoria, idc$departamento),
    c("departamento", "region", "idc_2025", "idc_2026", "diferencia_idc", "variacion_categoria")]


`subset()` filtra las filas que cumplen una condición. Veamos específicamente cuáles departamentos retrocedieron:

In [ ]:
subset(idc, variacion_categoria == "Retrocedió",
       select = c(departamento, idc_2025, idc_2026, diferencia_idc))


Y dónde está Norte de Santander en esta clasificación:

In [ ]:
idc[idc$departamento == "Norte de Santander",
    c("departamento", "idc_2025", "idc_2026", "diferencia_idc", "variacion_categoria")]


## 7. Gráficos: eligiendo la representación correcta según el tipo de variable

- **Gráfico de barras** y **de torta**: para variables **cualitativas** (categorías).
- **Histograma**: para variables **cuantitativas continuas** (valores numéricos agrupados en intervalos — el mismo procedimiento de la sección 3, pero dibujado en vez de tabulado).

### 7.1 Gráfico de barras — variable `region` (cualitativa nominal)

`barplot()` dibuja una barra por categoría, con altura proporcional a la frecuencia. `las = 2` rota las etiquetas del eje para que se puedan leer; `col` fija el color; `main` y `ylab` son el título y la etiqueta del eje vertical.

In [ ]:
barplot(f_region,
        las = 2,
        col = "#3B82F6",
        main = "Número de departamentos por región",
        ylab = "Frecuencia absoluta (f)")


### 7.2 Gráfico de barras y de torta — variable `variacion_categoria` (cualitativa ordinal)

Comparemos la misma información en dos formatos distintos. `pie()` dibuja un gráfico circular donde cada "porción" es proporcional a la frecuencia.

Para que cada porción muestre su **valor** (no solo el nombre de la categoría), construimos primero las etiquetas con `paste0()` — que pega texto y números en una sola cadena — combinando el nombre de la categoría, su frecuencia absoluta y su porcentaje.

In [ ]:
barplot(f_var,
        col = c("#EF4444", "#F59E0B", "#10B981"),
        main = "Departamentos según variación del IDC 2025-2026",
        ylab = "Frecuencia absoluta (f)")


In [ ]:
etiquetas_torta <- paste0(names(f_var), "\n", f_var, " (", round(100 * prop.table(f_var), 1), "%)")
etiquetas_torta


In [ ]:
pie(f_var,
    labels = etiquetas_torta,
    col = c("#EF4444", "#F59E0B", "#10B981"),
    main = "Departamentos según variación del IDC 2025-2026")


> **¿Barras o torta?** Cuando hay pocas categorías (2-4) y lo que importa es la *proporción del total*, la torta comunica bien — sobre todo si, como aquí, cada porción indica su propio valor. Cuando hay muchas categorías (como en `region`, con 8), o cuando además queremos comparar magnitudes exactas entre categorías, las barras suelen ser más claras de leer.

### 7.3 Histograma — variable `idc_2026` (cuantitativa continua)

`hist()` hace automáticamente lo mismo que construimos a mano en la sección 3: agrupa los valores en intervalos y dibuja la frecuencia de cada uno como una barra. Con `breaks = cortes` le pasamos exactamente los mismos límites de clase que ya calculamos, para que el gráfico coincida con la tabla.

In [ ]:
hist(idc$idc_2026,
     breaks = cortes,
     col = "#8B5CF6",
     main = "Distribución del IDC 2026 entre los departamentos",
     xlab = "IDC 2026",
     ylab = "Frecuencia absoluta (f)")


## 8. Para llevar

- `table()` calcula frecuencias absolutas; `prop.table()` las convierte en relativas; `cumsum()` las acumula.
- Para datos **cuantitativos continuos**, primero hay que agruparlos en intervalos (Rango, regla de Sturges, amplitud, `cut()`) antes de tabular — no se cuenta cada valor individual.
- La frecuencia acumulada solo tiene una interpretación plena cuando la variable tiene un **orden lógico** (nominal: no lo tiene; ordinal, discreta o continua agrupada: sí).
- `cut()` transforma una variable cuantitativa continua en una variable categórica, ya sea con puntos de corte técnicos (Sturges) o con puntos de corte definidos por el analista (como "Retrocedió/Estable/Mejoró").
- El tipo de gráfico depende del tipo de variable: barras/torta para cualitativas, histograma para cuantitativas continuas.

## Ejercicio propuesto

1. Repite la tabla de frecuencias por intervalos de la sección 3, pero para la variable `idc_2025` en vez de `idc_2026`. ¿El número de clases de Sturges cambia?
2. Crea una variable categórica similar a `variacion_categoria`, pero a partir de la diferencia en el pilar de **innovación** (`innovacion_2026 - innovacion_2025`). Usa los mismos puntos de corte (-0.05 y 0.05) o propón unos propios revisando primero el `summary()` de esa diferencia.
3. Construye su tabla de frecuencias completa (f, h, F, H) y grafícala con un gráfico de torta que muestre el valor de cada porción, igual que en la sección 7.2.

## Para la próxima clase

Vamos a calcular las **medidas de tendencia central** (media, mediana, moda) y de **dispersión** (varianza, desviación estándar, coeficiente de variación) sobre estas mismas variables.